# 🔭 Role 1 — Data Engineer & Preprocessor (v2 — Full Catalog)

Processes labeled TCEs from **Kepler + K2 + TESS** using official NASA catalogs.
Only downloads stars with known dispositions (confirmed / false positive / candidate)
so you get maximum training value per GB of disk used.

### Expected outputs
```
data/
├── processed/
│   ├── kepler/
│   │   └── {kepid}_processed.h5      ← one file per star
│   ├── k2/
│   │   └── {epicid}_processed.h5
│   └── tess/
│       └── {ticid}_processed.h5
├── augmented/
│   └── training_dataset.h5           ← X[N,2001], y[N] for Role 3
├── exports/
│   ├── master_catalog.csv            ← all targets with labels + h5 paths
│   └── role1_candidates.csv          ← successfully processed subset
└── plots/
    └── diagnostics/                  ← one detrending plot per target
```

### Interface contract for downstream roles
| Consumer | File | Keys |
|---|---|---|
| Role 2 | `processed/**/*.h5` | `stitched/time`, `stitched/flux`, `stitched/sector_ids`, `metadata` attrs |
| Role 3 | `augmented/training_dataset.h5` | `X[N,2001]` float32, `y[N]` int8 (0=NTP,1=planet,2=EB) |
| Role 4 | `exports/master_catalog.csv` | target_name, label, h5_path, noise_rms_ppm, mission |


## 0. Install & imports

In [ ]:
!pip install -q lightkurve wotan astropy astroquery h5py batman-package tqdm matplotlib scipy
# Restart runtime after this cell, then run from Cell 1 onwards

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py, os, warnings, time as time_module
from pathlib import Path
from tqdm import tqdm
import lightkurve as lk
from wotan import flatten
from astropy.stats import sigma_clip
import batman

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 100, 'axes.facecolor': '#0d1117',
    'figure.facecolor': '#0d1117', 'text.color': 'white',
    'axes.labelcolor': 'white', 'xtick.color': 'white',
    'ytick.color': 'white', 'axes.edgecolor': '#444'
})

# ── Directory structure ──────────────────────────────────────────────────────
for d in ['data/processed/kepler', 'data/processed/k2', 'data/processed/tess',
          'data/augmented', 'data/exports', 'data/plots/diagnostics']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✅ Ready')

---
## 1. Download Official NASA Labeled Catalogs

These catalogs tell us which stars have confirmed planets, false positives, or candidates.
We use them as our target list so every downloaded light curve has a ground-truth label.

| Catalog | Mission | Labels | Size |
|---|---|---|---|
| KOI cumulative | Kepler | CONFIRMED / FALSE POSITIVE / CANDIDATE | ~9500 TCEs |
| K2 candidates | K2 | CONFIRMED / FALSE POSITIVE / CANDIDATE | ~3000 TCEs |
| TOI catalog | TESS | KP / FP / PC / APC | ~7000 TCEs |

In [ ]:
# ── Kepler KOI catalog ────────────────────────────────────────────────────────
print('Downloading Kepler KOI catalog...')
koi_url = (
    'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI'
    '?table=cumulative&format=csv'
)
koi = pd.read_csv(koi_url, comment='#')

# Label mapping
koi['label'] = koi['koi_disposition'].map({
    'CONFIRMED':     1,
    'FALSE POSITIVE': 2,
    'CANDIDATE':     3,   # keep candidates — useful for semi-supervised
})
koi['mission'] = 'Kepler'
koi['target_name'] = 'KIC ' + koi['kepid'].astype(str)
koi['period'] = koi['koi_period']      # orbital period (days) — useful for Role 2
koi['depth_ppm'] = koi['koi_depth']    # transit depth in ppm
koi['duration_hr'] = koi['koi_duration']  # transit duration in hours

koi_clean = koi[['target_name', 'kepid', 'label', 'mission',
                  'koi_disposition', 'period', 'depth_ppm', 'duration_hr']].dropna(subset=['label'])

print(f'  KOI entries : {len(koi_clean)}')
print(koi_clean['koi_disposition'].value_counts().to_string())

In [ ]:
# ── K2 candidates catalog ─────────────────────────────────────────────────────
print('Downloading K2 catalog...')
k2_url = (
    'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI'
    '?table=k2candidates&format=csv'
)
try:
    k2 = pd.read_csv(k2_url, comment='#')
    k2['label'] = k2['disposition'].map({
        'CONFIRMED':     1,
        'FALSE POSITIVE': 2,
        'CANDIDATE':     3,
    })
    k2['mission'] = 'K2'
    k2['target_name'] = 'EPIC ' + k2['epic_name'].astype(str)
    k2['period'] = k2.get('pl_orbper', np.nan)
    k2['depth_ppm'] = k2.get('pl_trandep', np.nan)
    k2['duration_hr'] = k2.get('pl_trandur', np.nan)
    k2_clean = k2[['target_name', 'label', 'mission',
                    'disposition', 'period', 'depth_ppm', 'duration_hr']].dropna(subset=['label'])
    print(f'  K2 entries : {len(k2_clean)}')
    print(k2_clean['disposition'].value_counts().to_string())
except Exception as e:
    print(f'  ⚠️  K2 catalog fetch failed ({e}) — creating empty placeholder')
    k2_clean = pd.DataFrame(columns=['target_name', 'label', 'mission',
                                      'disposition', 'period', 'depth_ppm', 'duration_hr'])

In [ ]:
# ── TESS TOI catalog ──────────────────────────────────────────────────────────
print('Downloading TESS TOI catalog...')
toi_url = (
    'https://exoplanetarchive.ipac.caltech.edu/cgi-bin/nstedAPI/nph-nstedAPI'
    '?table=TOI&format=csv'
)
try:
    toi = pd.read_csv(toi_url, comment='#')
    toi['label'] = toi['tfopwg_disp'].map({
        'KP':  1,   # Known Planet
        'CP':  1,   # Confirmed Planet
        'FP':  2,   # False Positive
        'FA':  2,   # False Alarm
        'PC':  3,   # Planet Candidate
        'APC': 3,   # Ambiguous Planet Candidate
    })
    toi['mission'] = 'TESS'
    toi['target_name'] = 'TIC ' + toi['tid'].astype(str)
    toi['period'] = toi.get('pl_orbper', np.nan)
    toi['depth_ppm'] = toi.get('pl_trandep', np.nan)
    toi['duration_hr'] = toi.get('pl_trandur', np.nan)
    toi_clean = toi[['target_name', 'label', 'mission',
                      'tfopwg_disp', 'period', 'depth_ppm', 'duration_hr']].dropna(subset=['label'])
    print(f'  TOI entries : {len(toi_clean)}')
    print(toi_clean['tfopwg_disp'].value_counts().to_string())
except Exception as e:
    print(f'  ⚠️  TESS TOI catalog fetch failed ({e}) — creating empty placeholder')
    toi_clean = pd.DataFrame(columns=['target_name', 'label', 'mission',
                                       'tfopwg_disp', 'period', 'depth_ppm', 'duration_hr'])

In [ ]:
# ── Merge all catalogs into master list ───────────────────────────────────────
master = pd.concat([koi_clean, k2_clean, toi_clean], ignore_index=True)
master['h5_path'] = None       # filled in as we process each target
master['processed'] = False
master['noise_rms_ppm'] = np.nan
master['n_cadences'] = np.nan

master.to_csv('data/exports/master_catalog.csv', index=False)

print(f'\n📋 Master catalog: {len(master)} total TCEs')
print(master.groupby(['mission', 'label']).size().rename('count').to_string())
print('\nLabel key:  1=planet/confirmed  2=false positive  3=candidate')
print('Saved → data/exports/master_catalog.csv')

---
## 2. Core Processing Functions
Same as v1 — outlier removal, Wotan detrending, stitching, HDF5 save.
Fixed for newer astropy (no sigma_clip masked array bug).

In [ ]:
def remove_outliers(lc, sigma=5.0):
    lc = lc.remove_nans()
    try:
        lc = lc.remove_outliers(sigma=sigma)
    except Exception:
        pass
    flux = lc.flux.value.copy()
    median = np.nanmedian(flux)
    std = np.nanstd(flux)
    mask = np.abs(flux - median) < sigma * std
    return lc[mask]


def detrend(lc, method='biweight', window=0.5):
    time = lc.time.value
    flux = lc.flux.value / np.nanmedian(lc.flux.value)
    flat, trend = flatten(time, flux, method=method,
                          window_length=window, return_trend=True,
                          break_tolerance=0.5)
    return time, flat, trend, flux


def stitch(detrended_sectors):
    all_t, all_f, all_s = [], [], []
    for sid, s in enumerate(detrended_sectors):
        t, f = s['time'], s['flux'].copy()
        valid = np.isfinite(f)
        t, f = t[valid], f[valid]
        f = f / np.nanmedian(f)
        all_t.extend(t); all_f.extend(f); all_s.extend([sid]*len(t))
    idx = np.argsort(all_t)
    return np.array(all_t)[idx], np.array(all_f)[idx], np.array(all_s)[idx]


def save_hdf5(target_name, mission, time, flux, sector_ids,
              detrended_sectors, metadata):
    safe = target_name.replace(' ', '_').replace('-', '_')
    mission_dir = f'data/processed/{mission.lower()}'
    Path(mission_dir).mkdir(parents=True, exist_ok=True)
    out = Path(mission_dir) / f'{safe}_processed.h5'

    with h5py.File(out, 'w') as f:
        g = f.create_group('stitched')
        g.create_dataset('time',       data=time,       dtype='float64', compression='gzip')
        g.create_dataset('flux',       data=flux,       dtype='float64', compression='gzip')
        g.create_dataset('sector_ids', data=sector_ids, dtype='int16',   compression='gzip')
        gs = f.create_group('sectors')
        for i, s in enumerate(detrended_sectors):
            gg = gs.create_group(str(i))
            for k in ['time','flux','trend','raw_norm']:
                arr = s[k]; v = np.isfinite(arr)
                gg.create_dataset(k, data=arr[v], dtype='float64', compression='gzip')
        gm = f.create_group('metadata')
        for k, v in metadata.items():
            gm.attrs[k] = v
        gm.attrs['n_cadences'] = len(time)
        gm.attrs['t_start']    = float(time[0])
        gm.attrs['t_end']      = float(time[-1])
    return str(out)


def process_one_target(target_name, mission, max_sectors=4,
                        detrend_method='biweight', window=0.5,
                        plot_diagnostic=False):
    """
    Full pipeline for one target. Returns (h5_path, noise_rms_ppm, n_cadences)
    or raises on failure.
    """
    sr = lk.search_lightcurve(target_name, mission=mission, exptime='long')
    if len(sr) == 0:
        raise ValueError('No data found on MAST')

    lcc = sr[:max_sectors].download_all()
    lcs = [lcc[i] for i in range(len(lcc))]
    lcs = [remove_outliers(lc) for lc in lcs]

    det = []
    for lc in lcs:
        t, f, tr, rn = detrend(lc, method=detrend_method, window=window)
        det.append({'time': t, 'flux': f, 'trend': tr, 'raw_norm': rn})

    st_t, st_f, st_s = stitch(det)
    noise_rms = float(np.nanstd(st_f) * 1e6)

    meta = {
        'target_name': target_name, 'mission': mission,
        'detrend_method': detrend_method, 'window_length': window,
        'n_sectors': len(det)
    }
    h5 = save_hdf5(target_name, mission, st_t, st_f, st_s, det, meta)

    if plot_diagnostic:
        _plot_diagnostic(target_name, st_t, st_f, mission)

    return h5, noise_rms, len(st_t)


def _plot_diagnostic(name, time, flux, mission):
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(time, flux, 'o', ms=0.5, alpha=0.4, color='#00E5A0')
    ax.axhline(1.0, color='#FF6B6B', lw=0.7, ls='--')
    ax.set_title(f'{name} ({mission}) — stitched flat flux', fontsize=9)
    ax.set_xlabel('Time (days)'); ax.set_ylabel('Flux')
    safe = name.replace(' ','_')
    plt.tight_layout()
    plt.savefig(f'data/plots/diagnostics/{safe}.png', bbox_inches='tight', dpi=80)
    plt.close()


print('✅ Core functions ready')

---
## 3. Batch Processing — Kepler + K2 + TESS

### ⚠️ Disk & time budget
| Mission | N targets to process | Approx time | Approx disk |
|---|---|---|---|
| Kepler (confirmed only) | ~2300 | 4–6 hrs | ~15 GB |
| Kepler (confirmed + FP) | ~4500 | 8–12 hrs | ~30 GB |
| K2 confirmed | ~500 | 1–2 hrs | ~3 GB |
| TESS confirmed | ~400 | 1 hr | ~2 GB |

Colab free tier: ~80 GB disk, 12 hr session limit.
**Recommended: run Kepler confirmed + FP first (~4500 targets) — that alone is enough for Role 3.**

Use `BATCH_SIZE` and `START_IDX` to resume after a crash.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — tune these before running
# ─────────────────────────────────────────────────────────────────────────────

PROCESS_MISSIONS = ['Kepler', 'K2', 'TESS']   # remove any you want to skip

# Label filter: which dispositions to process
# 1=confirmed planet, 2=false positive, 3=candidate
# For training you want at minimum 1 and 2 (balanced classes)
LABEL_FILTER = [1, 2]   # skip candidates (label=3) to save time

MAX_SECTORS   = 4     # quarters/sectors per target
BATCH_SIZE    = 50    # process this many, then checkpoint
START_IDX     = 0     # change this to resume after a crash
PLOT_EVERY_N  = 20    # save a diagnostic plot every N targets (set 0 to disable)
SLEEP_BETWEEN = 1.0   # seconds between downloads (avoid MAST rate limits)

print('Config set. Run the next cell to start batch processing.')

In [ ]:
# Filter master catalog to targets we want to process
targets_to_process = master[
    (master['mission'].isin(PROCESS_MISSIONS)) &
    (master['label'].isin(LABEL_FILTER)) &
    (~master['processed'])
].copy().reset_index(drop=True)

print(f'Targets selected for processing: {len(targets_to_process)}')
print(targets_to_process.groupby(['mission','label']).size().rename('count').to_string())
print(f'\nStarting from index {START_IDX}, batch size {BATCH_SIZE}')
print(f'This batch: targets {START_IDX} → {START_IDX + BATCH_SIZE}')

In [ ]:
# ── MAIN BATCH LOOP ───────────────────────────────────────────────────────────
# Processes BATCH_SIZE targets, saves progress to CSV after each target.
# If Colab disconnects, change START_IDX and re-run this cell.

batch = targets_to_process.iloc[START_IDX : START_IDX + BATCH_SIZE]

success_count = 0
fail_count    = 0
fail_log      = []

for batch_i, (_, row) in enumerate(tqdm(batch.iterrows(),
                                         total=len(batch),
                                         desc='Processing')):
    global_idx = START_IDX + batch_i
    target     = row['target_name']
    mission    = row['mission']

    try:
        do_plot = (PLOT_EVERY_N > 0) and (batch_i % PLOT_EVERY_N == 0)
        h5, rms, n_cad = process_one_target(
            target, mission,
            max_sectors=MAX_SECTORS,
            plot_diagnostic=do_plot
        )

        # Update master catalog
        mask = master['target_name'] == target
        master.loc[mask, 'h5_path']       = h5
        master.loc[mask, 'processed']     = True
        master.loc[mask, 'noise_rms_ppm'] = rms
        master.loc[mask, 'n_cadences']    = n_cad

        success_count += 1

    except Exception as e:
        fail_count += 1
        fail_log.append({'target': target, 'mission': mission, 'error': str(e)})

    # Checkpoint every target — so crashes don't lose progress
    master.to_csv('data/exports/master_catalog.csv', index=False)
    time_module.sleep(SLEEP_BETWEEN)

print(f'\n✅ Batch done:  {success_count} succeeded  |  {fail_count} failed')
if fail_log:
    print('Failed targets:')
    for f in fail_log[:10]:
        print(f"  {f['target']} ({f['mission']}): {f['error']}")

In [ ]:
# ── Progress summary ──────────────────────────────────────────────────────────
done    = master[master['processed'] == True]
not_done = master[master['processed'] == False]

print('=== Processing progress ===')
print(f'  Done    : {len(done)}')
print(f'  Pending : {len(not_done)}')
print(f'  Total   : {len(master)}')

print('\nDone by mission + label:')
print(done.groupby(['mission','label']).size().rename('count').to_string())

print(f'\nNoise floor (ppm):')
print(done.groupby('mission')['noise_rms_ppm'].describe()[['mean','min','max']].to_string())

# To process the next batch:
next_start = START_IDX + BATCH_SIZE
print(f'\nTo continue: set START_IDX = {next_start} and re-run the batch cell')

---
## 4. Build Training Dataset for Role 3

Two sources combined:
1. **Real light curve segments** — extracted from processed HDF5 files, labeled from catalog
2. **Synthetic augmentation** — transit/EB injection on top of real noise

This is much stronger than v1 which only used augmentation from a single star.

In [ ]:
SEGMENT_LENGTH = 2001   # fixed cadence length for CNN input
SEGMENTS_PER_STAR = 3   # how many random segments to extract per processed star
SIGMA_NOISE = 300       # ppm — for augmented samples


def extract_segments_from_h5(h5_path, label, n_segments=SEGMENTS_PER_STAR,
                               seg_len=SEGMENT_LENGTH):
    """
    Load a processed HDF5, extract n_segments random fixed-length
    flux windows, return list of (flux_array, label).
    """
    with h5py.File(h5_path, 'r') as f:
        flux = f['stitched/flux'][:]

    flux = flux[np.isfinite(flux)]
    if len(flux) < seg_len:
        # Pad short light curves
        flux = np.pad(flux, (0, seg_len - len(flux)), mode='edge')

    segments = []
    for _ in range(n_segments):
        start = np.random.randint(0, max(1, len(flux) - seg_len))
        seg = flux[start:start + seg_len]
        # Normalize each segment to its own median
        seg = seg / np.nanmedian(seg)
        segments.append((seg.astype(np.float32), label))
    return segments


print('Segment extraction function ready.')

In [ ]:
# Augmentation helpers (same as v1)
def inject_gaussian_noise(flux, sigma_ppm=300):
    return flux + np.random.normal(0, sigma_ppm*1e-6, size=flux.shape)

def inject_red_noise(flux, amp=3e-4, tscale=30):
    w = np.random.normal(0, amp, len(flux))
    k = np.exp(-np.arange(tscale)/(tscale/3)); k /= k.sum()
    return flux + np.convolve(w, k, mode='same')

def simulate_blending(flux, bf=None):
    if bf is None: bf = np.random.uniform(0, 0.4)
    return (1-bf)*flux + bf*1.0, bf

def inject_transit(time, flux):
    period = np.random.uniform(1.0, 30.0)
    rp     = np.random.uniform(0.01, 0.15)
    a      = np.random.uniform(5.0, 50.0)
    inc    = np.random.uniform(85.0, 90.0)
    t0     = np.random.uniform(time[0], time[0]+period)
    p = batman.TransitParams()
    p.t0=t0; p.per=period; p.rp=rp; p.a=a; p.inc=inc
    p.ecc=0; p.w=90; p.limb_dark='quadratic'; p.u=[0.3,0.2]
    m = batman.TransitModel(p, time)
    return flux * m.light_curve(p)

def inject_eb(time, flux):
    period   = np.random.uniform(0.5, 20.0)
    depth    = np.random.uniform(0.005, 0.3)
    duration = np.random.uniform(0.05, 0.15)*period
    t0       = np.random.uniform(time[0], time[0]+period)
    ef = flux.copy()
    for tc in np.arange(t0, time[-1], period):
        tr = np.abs(time-tc)
        ie = tr < duration/2
        ef[ie] *= (1 - depth*(1 - tr[ie]/(duration/2)))
    return ef

print('Augmentation helpers ready.')

In [ ]:
# ── BUILD COMBINED DATASET ────────────────────────────────────────────────────
N_SYNTH_PER_CLASS = 500   # synthetic samples per class on top of real ones

X_all, y_all = [], []

# ── Part A: real labeled segments ─────────────────────────────────────────────
processed_done = master[master['processed'] == True].copy()
print(f'Extracting real segments from {len(processed_done)} processed stars...')

for _, row in tqdm(processed_done.iterrows(), total=len(processed_done), desc='Real segments'):
    try:
        segs = extract_segments_from_h5(row['h5_path'], int(row['label']))
        for seg, lbl in segs:
            X_all.append(seg)
            y_all.append(lbl)
    except Exception:
        pass

print(f'  Real segments extracted: {len(X_all)}')

# Use one of the processed stars as base for synthetic injection
# If nothing processed yet, fall back to flat baseline
if len(processed_done) > 0:
    base_h5 = processed_done.iloc[0]['h5_path']
    with h5py.File(base_h5, 'r') as f:
        base_flux = f['stitched/flux'][:]
        base_time = f['stitched/time'][:]
    base_flux = base_flux[np.isfinite(base_flux)]
    base_time = base_time[np.isfinite(base_flux[:len(base_time)])]
else:
    print('  No processed stars yet — using flat baseline for synthetics')
    base_time = np.linspace(0, 100, 10000)
    base_flux = np.ones(10000)


def get_seg(btime, bflux, length=SEGMENT_LENGTH):
    if len(bflux) < length:
        return np.linspace(0, length*0.02, length), np.ones(length)
    i = np.random.randint(0, len(bflux)-length)
    return btime[i:i+length], bflux[i:i+length].copy()


# ── Part B: synthetic NTP (label=0, map from label=3 or just noise) ───────────
# NOTE: We remap label 0 = NTP (non-transiting) for the CNN
# label mapping for training: 0=NTP, 1=planet, 2=EB
# (catalog labels: 1=confirmed, 2=FP, 3=candidate → CNN labels: 1, 2, skip/NTP)

print(f'Generating {N_SYNTH_PER_CLASS} synthetic NTP samples...')
np.random.seed(42)
for _ in range(N_SYNTH_PER_CLASS):
    _, sf = get_seg(base_time, base_flux)
    sf = inject_gaussian_noise(sf, SIGMA_NOISE)
    sf = inject_red_noise(sf)
    X_all.append(sf.astype(np.float32)); y_all.append(0)

print(f'Generating {N_SYNTH_PER_CLASS} synthetic planet transit samples...')
for _ in range(N_SYNTH_PER_CLASS):
    st, sf = get_seg(base_time, base_flux)
    sf = inject_gaussian_noise(sf, SIGMA_NOISE)
    sf = inject_transit(st, sf)
    if np.random.random() < 0.4:
        sf, _ = simulate_blending(sf)
    X_all.append(sf.astype(np.float32)); y_all.append(1)

print(f'Generating {N_SYNTH_PER_CLASS} synthetic EB samples...')
for _ in range(N_SYNTH_PER_CLASS):
    st, sf = get_seg(base_time, base_flux)
    sf = inject_gaussian_noise(sf, SIGMA_NOISE)
    sf = inject_eb(st, sf)
    X_all.append(sf.astype(np.float32)); y_all.append(2)

X_arr = np.array(X_all, dtype=np.float32)
y_arr = np.array(y_all, dtype=np.int8)

print(f'\n✅ Combined dataset: {X_arr.shape}')
print('Label counts:', {int(k): int(v) for k, v in zip(*np.unique(y_arr, return_counts=True))})
print('Label key: 0=NTP(non-transiting), 1=planet, 2=EB')

In [ ]:
# Save combined training dataset
aug_path = 'data/augmented/training_dataset.h5'
with h5py.File(aug_path, 'w') as f:
    f.create_dataset('X', data=X_arr, dtype='float32', compression='gzip')
    f.create_dataset('y', data=y_arr, dtype='int8')
    g = f.create_group('info')
    g.attrs['classes']         = '[0=NTP, 1=planet, 2=EB]'
    g.attrs['segment_length']  = SEGMENT_LENGTH
    g.attrs['n_total']         = len(y_arr)
    g.attrs['n_real']          = len(y_arr) - 3*N_SYNTH_PER_CLASS
    g.attrs['n_synthetic']     = 3*N_SYNTH_PER_CLASS
    g.attrs['sigma_noise_ppm'] = SIGMA_NOISE
    g.attrs['missions']        = str(PROCESS_MISSIONS)

size_mb = Path(aug_path).stat().st_size / 1e6
print(f'✅ Training dataset saved → {aug_path}  ({size_mb:.1f} MB)')
print(f'   Total samples : {len(y_arr)}')
print(f'   Real segments : {len(y_arr) - 3*N_SYNTH_PER_CLASS}')
print(f'   Synthetic     : {3*N_SYNTH_PER_CLASS}')

---
## 5. Final Export & Handoff

In [ ]:
# Export successfully processed targets for Role 4
role1_out = master[master['processed'] == True][[
    'target_name', 'mission', 'label', 'koi_disposition' if 'koi_disposition' in master.columns else 'label',
    'h5_path', 'n_cadences', 'noise_rms_ppm', 'period', 'depth_ppm'
]].copy()

role1_out.to_csv('data/exports/role1_candidates.csv', index=False)
print(f'✅ role1_candidates.csv → {len(role1_out)} processed targets')

print('\n=== FINAL HANDOFF SUMMARY ===')
print(f'Processed HDF5 files : {len(role1_out)}')
print(f'  Kepler             : {len(role1_out[role1_out["mission"]=="Kepler"])}')
print(f'  K2                 : {len(role1_out[role1_out["mission"]=="K2"])}')
print(f'  TESS               : {len(role1_out[role1_out["mission"]=="TESS"])}')
print(f'\nTraining dataset     : X={X_arr.shape}, y={y_arr.shape}')
print(f'\nFiles for each role:')
print(f'  Role 2 → data/processed/**/*.h5  (stitched/time, stitched/flux)')
print(f'  Role 3 → data/augmented/training_dataset.h5  (X, y)')
print(f'  Role 4 → data/exports/role1_candidates.csv')

In [ ]:
# Mount Google Drive and sync outputs (recommended over zip download for large runs)
from google.colab import drive
drive.mount('/content/drive')

import shutil
dest = '/content/drive/MyDrive/exoplanet_pipeline/role1_outputs'
Path(dest).mkdir(parents=True, exist_ok=True)

# Copy exports and augmented (small) — processed HDF5s are large, sync selectively
shutil.copytree('data/exports',   f'{dest}/exports',   dirs_exist_ok=True)
shutil.copytree('data/augmented', f'{dest}/augmented', dirs_exist_ok=True)
shutil.copytree('data/plots',     f'{dest}/plots',     dirs_exist_ok=True)

# For processed H5 files — copy only if total size is manageable
# shutil.copytree('data/processed', f'{dest}/processed', dirs_exist_ok=True)

print(f'✅ Outputs synced to Google Drive → {dest}')
print('Share the Drive folder link with Role 2, 3, and 4.')